# Data Creation Overview

This notebook builds the analysis dataset from the exported UK Biobank fields and reproduces the cohort-characteristics table (Table 1).

Starting from the post-exclusion cohort files in `./Num_0/`, it:
- merges the left/right eyes and both imaging instances into one record per participant;
- derives the demographic and systemic features (age, sex, education, diabetes, spherical equivalent, blood pressure, antihypertensive use, alcohol, smoking, BMI) and the OCT-derived retinal features (macular, RPE, disc, cup-to-disc and retinal-layer thicknesses);
- labels each participant as healthy / dementia / AD (with an AD-vascular-other split);
- computes descriptive statistics with t-tests and chi-square tests (`char_table`, `numeric_char_table`);
- saves distribution figures under `./results/images/`;
- writes the final labelled dataset `./results/df.csv`, consumed by `data_preprocessing.ipynb`.

## Prerequisite
The exported cohort CSVs under `./Num_0/` and `fields.py` (field-code to label definitions).

In [ ]:
import ast
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind

import fields

In [ ]:
df = pd.read_csv(
    r"./Num_0/All.csv", low_memory=False
)  # YT: All the data, after data exclusion
All = df.copy()
pd_D = pd.read_csv(
    r"./Num_0/D.csv", low_memory=False
)  # Dementia data, after data_exclusion
pd_AD = pd.read_csv(
    r"./Num_0/AD.csv", low_memory=False
)  # AD_data, after data_exclusion
OCTs = []
df.info()

In [ ]:
# df_D has 500 rows and 401 columns
pd_D.info()

In [ ]:
# df_AD has 223 rows and 401 columns
pd_AD.info()

In [ ]:
"""
OCT_0, selected instance 0, left eye after data_exclusion
OCT_1, selected instance 0, right eye after data_exclusion
OCT_2, selected instance 1, left eye after data_exclusion
OCT_3, selected instance 1, right eye after data_exclusion
"""

for i in range(4):
    OCT = pd.read_csv(r"./Num_0/OCT" + str(i) + ".csv", low_memory=False)
    OCT = OCT.drop(OCT.columns[0], axis=1)
    OCTs.append(OCT)


def join_pds(pd0, pd1):
    """
    This function combines left and right eye data for two instances without duplicates.
    """
    filtered_All = pd1[~pd1["eid"].isin(pd0["eid"])]
    joined = pd.concat([pd0, filtered_All], ignore_index=True)
    return joined


OCT_0 = join_pds(OCTs[0], OCTs[1])  # YT: Instance 0 cohorts
OCT_1 = join_pds(OCTs[2], OCTs[3])  # YT: Instance 1 cohorts

In [ ]:
OCT_0.info()

In [ ]:
OCT_1.info()

In [ ]:
Inst0_d = pd.read_csv(
    r"./Num_0/Demog_0.csv"
)  # YT Demographic data of selected instance 0 cohorts
Inst1_d = pd.read_csv(
    r"./Num_0/Demog_1.csv"
)  # YT Demographic data of selected instance 1 cohorts

Inst0_n = pd.read_csv(
    r"./Num_0/Num_0.csv"
)  # YT Numerical OCT data of selected instance 0 cohorts
Inst1_n = pd.read_csv(
    r"./Num_0/Num_1.csv"
)  # YT Numerical OCT data of selected instance 1 cohorts

In [ ]:
d0 = Inst0_d[["eid", "p53_i0"]].copy()
d1 = Inst1_d[["eid", "p53_i0"]].copy()
filter = ~d1["eid"].isin(d0["eid"])  # YT: Check if the eids in d1 are also in d0
d1 = d1[filter]  # YT: Filter out the eids in d1 that are also in d0
d0 = d0.rename(columns={"p53_i0": "examination year"})
d1 = d1.rename(columns={"p53_i0": "examination year"})

os.makedirs("./results", exist_ok=True)

# YT: Merge the two dataframes on 'eid' and 'Death Year'
cohort_death_years = pd.concat([d0, d1], ignore_index=True)
cohort_death_years.to_csv(r"./results/cohort_examination_years.csv", index=False)

In [ ]:
Char_Table = {
    "Main_Char": [""],
    "Sub_Char": [""],
    "N": [len(df)],
    "Overall": [len(df)],
    "Stat O": [0],
    "w/o Dementia": [len(df) - len(pd_D)],
    "Stat w/o Dementia": [0],
    "Dementia": [len(pd_D)],
    "Stat Dementia": [0],
    "AD": [len(pd_AD)],
    "Stat AD": [0],
    "t-value Dementia": [0],
    "chi2 Dementia": [0],
    "p-value Dementia": [0],
    "t-value AD": [0],
    "chi2 AD": [0],
    "p-value AD": [0],
}

Numeric_Char_Table = {
    "Main_Char": [""],
    "Sub_Char": [""],
    "N": [len(df)],
    "Overall": [len(df)],
    "Stat O": [0],
    "w/o Dementia": [len(df) - len(pd_D)],
    "Stat w/o Dementia": [0],
    "Dementia": [len(pd_D)],
    "Stat Dementia": [0],
    "AD": [len(pd_AD)],
    "Stat AD": [0],
    "t-value Dementia": [0],
    "p-value Dementia": [0],
    "t-value AD": [0],
    "p-value AD": [0],
}


# Returns count, mean, and standard deviation of selected columns
def GetRowMeanData(df, pList):
    count = len(df)
    mean_values = df[pList].mean()
    std_values = df[pList].std()
    return count, mean_values, std_values


# Returns count, median, and interquartile range (25th and 75th percentiles) of selected columns
def GetRowIQRData(df, pList):
    count = len(df)
    median_values = df[pList].median()
    iqr_25 = df[pList].quantile(0.25).round(2)
    iqr_75 = df[pList].quantile(0.75).round(2)
    return count, median_values, iqr_25, iqr_75


# Returns count and percentage of selected columns based on the row means
def GetRowPercData(df, pList):
    count = len(df)
    perc_values = (df[pList].mean() * 100).round(2)  # Convert to percentages
    return count, perc_values


# Adds a new row to the Char_Table with specified statistics and test results
def AddRow(
    sMain,
    sSub,
    nN,
    meano,
    sto,
    meanWO,
    stdWO,
    meanW,
    stdW,
    meanAD,
    stdAD,
    td="-",
    cd="-",
    _pd="-",
    tad="-",
    cad="-",
    pad="-",
):
    global Char_Table

    Char_Table["Main_Char"].append(sMain)
    Char_Table["Sub_Char"].append(sSub)
    Char_Table["N"].append(nN)
    Char_Table["Overall"].append(meano)
    Char_Table["Stat O"].append(sto)
    Char_Table["w/o Dementia"].append(meanWO)
    Char_Table["Stat w/o Dementia"].append(stdWO)
    Char_Table["Dementia"].append(meanW)
    Char_Table["Stat Dementia"].append(stdW)
    Char_Table["AD"].append(meanAD)
    Char_Table["Stat AD"].append(stdAD)
    Char_Table["t-value Dementia"].append(td)
    Char_Table["chi2 Dementia"].append(cd)
    Char_Table["p-value Dementia"].append(_pd)
    Char_Table["t-value AD"].append(tad)
    Char_Table["chi2 AD"].append(cad)
    Char_Table["p-value AD"].append(pad)


# Adds a new row to the Numeric_Char_Table with specified statistics and test results
def AddRow_Numeric(
    sMain,
    sSub,
    nN,
    meano,
    sto,
    meanWO,
    stdWO,
    meanW,
    stdW,
    meanAD,
    stdAD,
    td="-",
    _pd="-",
    tad="-",
    pad="-",
):
    global Numeric_Char_Table

    Numeric_Char_Table["Main_Char"].append(sMain)
    Numeric_Char_Table["Sub_Char"].append(sSub)
    Numeric_Char_Table["N"].append(nN)
    Numeric_Char_Table["Overall"].append(meano)
    Numeric_Char_Table["Stat O"].append(sto)
    Numeric_Char_Table["w/o Dementia"].append(meanWO)
    Numeric_Char_Table["Stat w/o Dementia"].append(stdWO)
    Numeric_Char_Table["Dementia"].append(meanW)
    Numeric_Char_Table["Stat Dementia"].append(stdW)
    Numeric_Char_Table["AD"].append(meanAD)
    Numeric_Char_Table["Stat AD"].append(stdAD)
    Numeric_Char_Table["t-value Dementia"].append(td)
    Numeric_Char_Table["p-value Dementia"].append(_pd)
    Numeric_Char_Table["t-value AD"].append(tad)
    Numeric_Char_Table["p-value AD"].append(pad)


def GetWithoutWithD(_pd):
    """Splits data into two subsets: one excluding and one including entries in pd_D based on 'eid'."""
    without_dementia = _pd[~_pd["eid"].isin(pd_D["eid"])]
    with_dementia = _pd[_pd["eid"].isin(pd_D["eid"])]
    return without_dementia, with_dementia


def GetAD(_pd):
    """Returns a subset of the data including only entries present in pd_AD based on 'eid'."""
    return _pd[_pd["eid"].isin(pd_AD["eid"])]


def check_value(cell, val):
    """Helper function to check if val is in cell, for cells that may contain lists or single values."""
    return val in cell if isinstance(cell, list) else cell == val


# Perform chi-square tests
def perform_chi_square(counts_a, counts_b, label, index):
    """Helper function to perform chi-square and display results."""
    contingency_table = pd.DataFrame(
        {label: counts_a, "No " + label: counts_b}, index=index
    )
    chi2, p_val, _, expected = chi2_contingency(contingency_table)
    print(f"{label} Contingency Table:\n{contingency_table}")
    print(
        f"Chi-Square Statistic: {chi2}\np-value: {p_val}\nExpected Frequencies:\n{pd.DataFrame(expected, columns=contingency_table.columns, index=contingency_table.index)}\n"
    )
    return chi2, p_val


def Add_Age(sMain, sSub):
    """
    Calculates age-related statistics and adds them to Char_Table.

    Parameters:
    sMain : str
        The main characteristic name to add to Char_Table.
    sSub : str
        The sub-characteristic name associated with the age category.
    """
    global All, OCTs, ppd, pd_D, pd_AD, OCT1, OCT0, Char_Table

    # Calculate age for both OCT0 and OCT1 groups
    Age0, Age1 = OCT_0.copy(), OCT_1.copy()
    Age0["Age"] = pd.to_datetime(Age0["p53_i0"]).dt.year - Age0["p34"]
    Age1["Age"] = pd.to_datetime(Age1["p53_i1"]).dt.year - Age1["p34"]

    # Combine ages, ensuring non-overlapping records from Age1
    AgeAll = (
        pd.concat([Age0, Age1[~Age1["eid"].isin(Age0["eid"])]], ignore_index=True)
        .loc[:, ["eid", "Age"]]
        .dropna(subset=["Age"])
    )
    nN = len(AgeAll)

    # Overall statistics
    nOverall, mean_overall, std_overall = GetRowMeanData(AgeAll, "Age")

    # Dementia subsets: without dementia, with dementia, and with AD
    without_dementia, with_dementia = GetWithoutWithD(AgeAll)
    with_ad = GetAD(AgeAll)

    # Calculate statistics for each subset
    _, mean_wo, std_wo = GetRowMeanData(without_dementia, "Age")
    _, mean_w, std_w = GetRowMeanData(with_dementia, "Age")
    _, mean_ad, std_ad = GetRowMeanData(with_ad, "Age")

    # Perform t-tests for statistical comparisons
    t_stat_d, p_value_d = ttest_ind(
        without_dementia["Age"], with_dementia["Age"], equal_var=False
    )
    t_stat_ad, p_value_ad = ttest_ind(
        without_dementia["Age"], with_ad["Age"], equal_var=False
    )

    # Add calculated data to Char_Table
    AddRow(
        sMain,
        sSub,
        nN,
        mean_overall,
        std_overall,
        mean_wo,
        std_wo,
        mean_w,
        std_w,
        mean_ad,
        std_ad,
        td=t_stat_d,
        _pd=p_value_d,
        tad=t_stat_ad,
        pad=p_value_ad,
    )
    return without_dementia, with_dementia, with_ad


def Add_Category(sMain, sSub, col, Vals):
    """
    Calculates category-based statistics for a specified column and adds results to Char_Table.
    Also performs chi-square tests of independence between dementia status and category values.
    """

    # Filter data and initialize variables
    pdf = All.dropna(subset=[col]).copy()
    nN = len(pdf)
    Counts, Sub_wo, Sub_w, Sub_AD = [], [], [], []
    return_wo, return_w, return_AD = [], [], []
    dementia_counts, no_dementia_counts, AD_counts = [], [], []

    # Calculate counts for each category and prepare chi-square data
    for val in Vals:
        sub = pdf[pdf[col] == val].loc[:, ["eid", col]].dropna(subset=[col])
        Counts.append(len(sub))

        sub_wo, sub_w = GetWithoutWithD(sub)
        Sub_wo.append(len(sub_wo))
        Sub_w.append(len(sub_w))

        sub_AD = GetAD(sub)
        Sub_AD.append(len(sub_AD))

        dementia_counts.append(len(sub_w))
        no_dementia_counts.append(len(sub_wo))
        AD_counts.append(len(sub_AD))
        return_wo.append(sub_wo)
        return_w.append(sub_w)
        return_AD.append(sub_AD)

    # Calculate percentages
    total_counts = sum(Counts)
    total_wo = sum(Sub_wo)
    total_w = sum(Sub_w)
    total_ad = sum(Sub_AD)

    perc = [count * 100 / total_counts for count in Counts]
    perc_wo = [count * 100 / total_wo if total_wo else 0 for count in Sub_wo]
    perc_w = [count * 100 / total_w if total_w else 0 for count in Sub_w]
    perc_AD = [count * 100 / total_ad if total_ad else 0 for count in Sub_AD]

    # Perform chi-square tests
    # Dementia chi-square test
    contingency_table_d = pd.DataFrame(
        {"Dementia": dementia_counts, "No Dementia": no_dementia_counts}, index=sSub
    )
    chi2_d, p_d, _, expected_d = chi2_contingency(contingency_table_d)

    # AD chi-square test
    contingency_table_ad = pd.DataFrame(
        {"AD": AD_counts, "No AD": no_dementia_counts}, index=sSub
    )
    chi2_ad, p_ad, _, expected_ad = chi2_contingency(contingency_table_ad)

    # Display contingency tables and chi-square results
    for label, table, chi2, p_val, expected in [
        ("Dementia", contingency_table_d, chi2_d, p_d, expected_d),
        ("AD", contingency_table_ad, chi2_ad, p_ad, expected_ad),
    ]:
        print(f"{label} Contingency Table:\n{table}")
        print(f"Chi-Square Statistic: {chi2}")
        print(f"p-value: {p_val}")
        print(
            f"Expected Frequencies:\n{pd.DataFrame(expected, columns=table.columns, index=table.index)}\n"
        )

    # Add results to Char_Table
    for i, val in enumerate(Vals):
        AddRow(
            sMain,
            sSub[i],
            nN,
            Counts[i],
            perc[i],
            Sub_wo[i],
            perc_wo[i],
            Sub_w[i],
            perc_w[i],
            Sub_AD[i],
            perc_AD[i],
            cd=chi2_d,
            _pd=p_d,
            cad=chi2_ad,
            pad=p_ad,
        )
    return return_wo, return_w, return_AD


def Add_Category_2_Inst(sMain, sSub, cols, Vals):
    """
    Calculates category-based statistics from two separate data instances and adds results to Char_Table.
    Also performs chi-square tests of independence between dementia/AD status and category values.
    """

    Counts, Sub_wo, Sub_w, Sub_AD = [], [], [], []
    dementia_counts, no_dementia_counts, AD_counts = [], [], []
    return_wo, return_w, return_AD = [], [], []

    # Calculate counts for each category across instances and filter subsets
    for i, val in enumerate(Vals):
        # Filter by value in each instance and ensure unique `eid` per subject
        sub0 = Inst0_d[Inst0_d[cols[0]].eq(val).any(axis=1)]
        sub1 = Inst1_d[Inst1_d[cols[1]].eq(val).any(axis=1)]
        filtered_All = sub1[~sub1["eid"].isin(sub0["eid"])]
        sub = (
            pd.concat([sub0, filtered_All], ignore_index=True)
            .loc[:, ["eid", cols[0][0]]]
            .dropna()
        )
        sub[cols[0][0]] = i

        Counts.append(len(sub))

        # Split based on dementia status
        sub_wo, sub_w = GetWithoutWithD(sub)
        Sub_wo.append(len(sub_wo))
        Sub_w.append(len(sub_w))

        # Subset with AD status
        sub_AD = GetAD(sub)
        Sub_AD.append(len(sub_AD))

        # Append counts for chi-square analysis
        dementia_counts.append(len(sub_w))
        no_dementia_counts.append(len(sub_wo))
        AD_counts.append(len(sub_AD))
        return_wo.append(sub_wo)
        return_w.append(sub_w)
        return_AD.append(sub_AD)
    # Calculate percentages for each subset
    total_counts = sum(Counts)
    total_wo, total_w, total_ad = sum(Sub_wo), sum(Sub_w), sum(Sub_AD)
    perc = [count * 100 / total_counts for count in Counts]
    perc_wo = [count * 100 / total_wo if total_wo else 0 for count in Sub_wo]
    perc_w = [count * 100 / total_w if total_w else 0 for count in Sub_w]
    perc_AD = [count * 100 / total_ad if total_ad else 0 for count in Sub_AD]

    # Dementia chi-square
    chi2_d, p_d = perform_chi_square(
        dementia_counts, no_dementia_counts, "Dementia", sSub
    )

    # AD chi-square
    chi2_ad, p_ad = perform_chi_square(AD_counts, no_dementia_counts, "AD", sSub)

    # Add results to Char_Table
    for i, val in enumerate(Vals):
        AddRow(
            sMain,
            sSub[i],
            total_counts,
            Counts[i],
            perc[i],
            Sub_wo[i],
            perc_wo[i],
            Sub_w[i],
            perc_w[i],
            Sub_AD[i],
            perc_AD[i],
            cd=chi2_d,
            _pd=p_d,
            cad=chi2_ad,
            pad=p_ad,
        )
    return return_wo, return_w, return_AD


def Add_Category_2_InstMean(Ins0, Ins1, sMain, sSub, cols, Vals):
    """
    Calculates mean-based statistics for a specific characteristic across two instances.
    """

    # Create copies of the DataFrames and initialize 'New' column for analysis
    Pd0 = Ins0.copy()
    Pd1 = Ins1.copy()
    Pd0["New"] = Pd0[cols[0]]
    Pd1["New"] = Pd1[cols[1]]

    # Filter out the rows in Pd1 that are already present in Pd0
    filtered_All = Pd1[~Pd1["eid"].isin(Pd0["eid"])]

    # Concatenate and clean data
    pdAll = (
        pd.concat([Pd0, filtered_All], ignore_index=True)
        .loc[:, ["eid", "New"]]
        .dropna(subset=["New"])
    )

    # Total number of entries after filtering and combining
    nN = len(pdAll)

    # Calculate means and standard deviations for different subsets
    nOverAll, meano, sto = GetRowMeanData(pdAll, "New")
    sub_wo, sub_w = GetWithoutWithD(pdAll)
    nW, meanW, stdW = GetRowMeanData(sub_w, "New")
    nWO, meanWO, stdWO = GetRowMeanData(sub_wo, "New")
    sub_AD = GetAD(pdAll)
    nAD, meanAD, stdAD = GetRowMeanData(sub_AD, "New")

    # Perform statistical tests (t-tests) for dementia and AD
    t_stat_d, p_value_d = ttest_ind(sub_wo["New"], sub_w["New"], equal_var=False)
    t_stat_ad, p_value_ad = ttest_ind(sub_wo["New"], sub_AD["New"], equal_var=False)

    # Add results to the Char_Table using the AddRow function
    AddRow(
        sMain,
        sSub,
        nN,
        meano,
        sto,
        meanWO,
        stdWO,
        meanW,
        stdW,
        meanAD,
        stdAD,
        td=t_stat_d,
        _pd=p_value_d,
        tad=t_stat_ad,
        pad=p_value_ad,
    )
    return sub_wo, sub_w, sub_AD


def Add_Category_2_InstIQR(Ins0, Ins1, sMain, sSub, cols):
    """
    Calculates IQR-based statistics for a specific characteristic across two instances.
    """

    global All, OCTs, ppd, pd_D, pd_AD, OCT1, OCT0, Char_Table

    Pd0 = Ins0.copy()
    Pd1 = Ins1.copy()
    Pd0["New"] = Pd0[cols[0]]
    Pd1["New"] = Pd1[cols[1]]

    filtered_All = Pd1[~Pd1["eid"].isin(Pd0["eid"])]
    pdAll = (
        pd.concat([Pd0, filtered_All], axis=0, ignore_index=True)
        .loc[:, ["eid", "New"]]
        .dropna(subset=["New"])
    )

    nN = len(pdAll)

    nOverAll, mediano, iqr1o, iqr3o = GetRowIQRData(pdAll, "New")
    stato = f"({iqr1o}-{iqr3o})"
    sub_wo, sub_w = GetWithoutWithD(pdAll)
    nW, medianW, iqr1W, iqr3W = GetRowIQRData(sub_w, "New")
    statW = f"({iqr1W}-{iqr3W})"
    nWO, medianWO, iqr1WO, iqr3WO = GetRowIQRData(sub_wo, "New")
    statWO = f"({iqr1WO}-{iqr3WO})"
    sub_AD = GetAD(pdAll)
    nAD, medianAD, iqr1AD, iqr3AD = GetRowIQRData(sub_AD, "New")
    statAD = f"({iqr1AD}-{iqr3AD})"
    t_stat_d, p_value_d = ttest_ind(sub_wo["New"], sub_w["New"], equal_var=False)
    t_stat_ad, p_value_ad = ttest_ind(sub_wo["New"], sub_AD["New"], equal_var=False)
    AddRow(
        sMain,
        sSub,
        nN,
        mediano,
        stato,
        medianWO,
        statWO,
        medianW,
        statW,
        medianAD,
        statAD,
        td=t_stat_d,
        _pd=p_value_d,
        tad=t_stat_ad,
        pad=p_value_ad,
    )
    return sub_wo, sub_w, sub_AD


def Add_Category_multiple(sMain, sSub, cols, Vals, insts0, insts1):
    """
    Calculates category-based statistics from multiple instances of data.
    """

    Counts, Sub_wo, Sub_w, Sub_AD = [], [], [], []
    return_wo, return_w, return_AD = [], [], []

    # Temporary lists for constructing the chi-square contingency table
    dementia_counts = []
    no_dementia_counts = []
    AD_counts = []

    # Ensure sSub and Vals lengths match for correct pairing
    if len(sSub) != len(Vals):
        raise ValueError("Length of sSub and Vals must match.")

    for i, val_set in enumerate(Vals):
        # Filter insts0 and insts1 based on Vals[i]
        sub0_filters = insts0[cols[0][0]].isin(val_set[0])
        sub1_filters = insts1[cols[1][0]].isin(val_set[0])

        sub0 = (
            insts0[sub0_filters].loc[:, ["eid", cols[0][0]]].dropna(subset=[cols[0][0]])
        )
        sub1 = (
            insts1[sub1_filters].loc[:, ["eid", cols[1][0]]].dropna(subset=[cols[1][0]])
        )
        sub0.columns = ["eid", "New"]
        sub1.columns = ["eid", "New"]

        # Filter and combine subsets (instances)
        filtered_All = sub1[~sub1["eid"].isin(sub0["eid"])]
        sub = pd.concat([sub0, filtered_All], axis=0, ignore_index=True)
        sub["New"] = i

        # Calculate counts and append to respective lists
        Counts.append(len(sub))
        sub_wo, sub_w = GetWithoutWithD(sub)
        sub_ad = GetAD(sub)
        Sub_wo.append(len(sub_wo))
        Sub_w.append(len(sub_w))
        Sub_AD.append(len(sub_ad))

        # Append counts for chi-square contingency table
        dementia_counts.append(len(sub_w))
        no_dementia_counts.append(len(sub_wo))
        AD_counts.append(len(sub_ad))
        return_wo.append(sub_wo)
        return_w.append(sub_w)
        return_AD.append(sub_ad)

    # Totals for calculations
    nN = sum(Counts)
    total_Sub_wo = sum(Sub_wo)
    total_Sub_w = sum(Sub_w)
    total_Sub_AD = sum(Sub_AD)

    # Calculate percentages and add the row
    perc = []
    perc_wo = []
    perc_w = []
    perc_AD = []

    for i in range(len(Vals)):
        perc.append((Counts[i] * 100 / nN) if nN > 0 else 0)
        perc_wo.append((Sub_wo[i] * 100 / total_Sub_wo) if total_Sub_wo > 0 else 0)
        perc_w.append((Sub_w[i] * 100 / total_Sub_w) if total_Sub_w > 0 else 0)
        perc_AD.append((Sub_AD[i] * 100 / total_Sub_AD) if total_Sub_AD > 0 else 0)

    # Perform chi-square tests
    chi2_d, p_d = perform_chi_square(
        dementia_counts, no_dementia_counts, "Dementia", sSub
    )
    chi2_ad, p_ad = perform_chi_square(AD_counts, no_dementia_counts, "AD", sSub)

    # Add results to Char_Table
    for i in range(len(Vals)):
        AddRow(
            sMain,
            sSub[i],
            nN,
            Counts[i],
            perc[i],
            Sub_wo[i],
            perc_wo[i],
            Sub_w[i],
            perc_w[i],
            Sub_AD[i],
            perc_AD[i],
            cd=chi2_d,
            _pd=p_d,
            cad=chi2_ad,
            pad=p_ad,
        )
    return return_wo, return_w, return_AD


def check_diabetes_man(cell):
    """Helper function to check if diabetes medication is used"""
    if isinstance(cell, list) and cell:
        # existance of 3 means usage of insulin
        if 3 in cell:
            return 1
        elif 1 in cell or 2 in cell or -7 in cell:
            return 0
    return np.nan


def check_diabetes_woman(cell):
    """Helper function to check if diabetes medication is used"""
    if isinstance(cell, list) and cell:
        # existance of 3 means usage of insulin
        if 3 in cell:
            return 1
        elif 1 in cell or 2 in cell or -7 in cell or 4 in cell or 5 in cell:
            return 0
    return np.nan


def check_hypertension_man(cell):
    """Helper function to check if hypertension medication is used for man"""
    if isinstance(cell, list) and cell:
        # existance of 2 means usage of antihypertensive medication
        if 2 in cell:
            return 1
        elif 1 in cell or 3 in cell or -7 in cell:
            return 0
    return np.nan


def check_hypertension_woman(cell):
    """Helper function to check if hypertension medication is used for woman"""
    if isinstance(cell, list) and cell:
        # existance of 2 means usage of antihypertensive medication
        if 2 in cell:
            return 1
        elif 1 in cell or 3 in cell or -7 in cell or 4 in cell or 5 in cell:
            return 0
    return np.nan


def combine_left_right(df, left_col_id, right_col_id):

    avg_df = pd.DataFrame()
    # first column is going to bo eid column, we might change it to index later on
    avg_df["eid"] = df["eid"]

    avg_df[f"{left_col_id}_mean"] = np.where(
        # both left and right exists
        df[left_col_id].notna() & df[right_col_id].notna(),
        (df[left_col_id] + df[right_col_id]) / 2,
        np.where(
            # only left exists
            df[left_col_id].notna(),
            df[left_col_id],
            np.where(
                # only right exists
                df[right_col_id].notna(),
                df[right_col_id],
                # no data
                np.nan,
            ),
        ),
    )
    return avg_df


def combine_diabetes(df1, col1, col2):
    """
    Combines diabetes data from two dataframes.
    Returns 1 if either source indicates diabetes, 0 if at least one source
    says no diabetes, and NaN if both sources are missing.
    """
    has_diabetes = (df1[col1] == 1) | (df1[col2] == 1)
    both_missing = df1[col1].isna() & df1[col2].isna()
    result = has_diabetes.astype(float)  # float to support NaN
    result[both_missing] = np.nan
    return result


def Add_Category_disease(sMain, sSub, dfs):
    """
    Add_Category function calculates category-based statistics
    """

    global All
    nN = len(All)
    # Counts of entries in pdf for each category.
    Counts = []
    # Counts of entries in each category that are not in pd_D.
    Sub_wo = []
    # Counts of entries in each category that are in pd_D.
    Sub_w = []
    # cOUNTS OF ENTRIES IN EACH CATEGORY THAT ARE IN pd_AD
    Sub_AD = []

    # calculate counts
    for i in range(len(dfs)):
        sub = dfs[i]
        Counts.append(len(sub))
        sub_wo, sub_w = GetWithoutWithD(sub)
        Sub_wo.append(len(sub_wo))
        Sub_w.append(len(sub_w))
        sub_AD = GetAD(sub)
        Sub_AD.append(len(sub_AD))

    # calculate percantages and add results to Char_Table
    for i in range(len(dfs)):
        perc = Counts[i] * 100 / nN

        if Sub_wo[i] == 0:
            perc_wo = 0
        else:
            perc_wo = Sub_wo[i] * 100 / nN
        perc_w = Sub_w[i] * 100 / nN
        perc_AD = Sub_AD[i] * 100 / nN
        AddRow(
            sMain,
            sSub[i],
            nN,
            Counts[i],
            perc,
            Sub_wo[i],
            perc_wo,
            Sub_w[i],
            perc_w,
            Sub_AD[i],
            perc_AD,
        )


def calculate_mean_numerical_features(df, column_names):
    """
    calculate_mean_numerical_features function calculates mean of the numerical features. It uses mean of the left and right eye values if
    both of them exists. If only one of them exists, it uses that value. If none of them exists, it uses NaN.
    """

    avg_df = pd.DataFrame()
    # first column is going to bo eid column, we might change it to index later on
    avg_df["eid"] = df["eid"]

    columns = [col for col in df.columns if col != "eid"]

    # subsequent columns represents left and right eyes respectively
    for i in range(0, len(columns), 2):
        left_col_id = columns[i]
        right_col_id = columns[i + 1]

        avg_df[columns[i]] = np.where(
            # both left and right exists
            df[left_col_id].notna() & df[right_col_id].notna(),
            (df[left_col_id] + df[right_col_id]) / 2,
            np.where(
                # only left exists
                df[left_col_id].notna(),
                df[left_col_id],
                np.where(
                    # only right exists
                    df[right_col_id].notna(),
                    df[right_col_id],
                    # no data
                    np.nan,
                ),
            ),
        )

    # change the column names
    avg_df.columns = column_names
    return avg_df


def Add_numeric_mean(sMain, sSub, cols, df, func):
    """
    Add_numeric_mean function calculates mean-based statistics for a numeric eye characteristics.
    """

    if len(sSub) != len(cols):
        raise ValueError("Length of sSub and Vals must match.")

    return_wo, return_w, return_AD = [], [], []

    for i in range(len(cols)):
        sub = df.loc[:, ["eid", cols[i]]].dropna(subset=[cols[i]])
        nOverAll, meano, sto = GetRowMeanData(sub, cols[i])
        sub_wo, sub_w = GetWithoutWithD(sub)
        nW, meanW, stdW = GetRowMeanData(sub_w, cols[i])
        nWO, meanWO, stdWO = GetRowMeanData(sub_wo, cols[i])
        sub_AD = GetAD(sub)
        nAD, meanAD, stdAD = GetRowMeanData(sub_AD, cols[i])
        return_wo.append(sub_wo)
        return_w.append(sub_w)
        return_AD.append(sub_AD)

        t_stat_d, p_value_d = ttest_ind(
            sub_wo[cols[i]], sub_w[cols[i]], equal_var=False
        )
        t_stat_ad, p_value_ad = ttest_ind(
            sub_wo[cols[i]], sub_AD[cols[i]], equal_var=False
        )

        func(
            sMain,
            sSub[i],
            nOverAll,
            meano,
            sto,
            meanWO,
            stdWO,
            meanW,
            stdW,
            meanAD,
            stdAD,
            td=t_stat_d,
            _pd=p_value_d,
            tad=t_stat_ad,
            pad=p_value_ad,
        )
    return return_wo, return_w, return_AD


def SaveTableAsCSV(pTable, file_name="output.csv"):
    df = pd.DataFrame(pTable)

    # set a default float format for other columns
    pd.options.display.float_format = "{:.2f}".format

    # check if "p-value Dementia" column exists and convert it to numeric if necessary
    if "p-value Dementia" in df.columns:
        df["p-value Dementia"] = pd.to_numeric(df["p-value Dementia"], errors="coerce")
        # apply scientific notation for very small p-values
        df["p-value Dementia"] = df["p-value Dementia"].apply(
            lambda x: f"{x:.0e}" if x < 1e-3 else f"{x:.3f}" if pd.notnull(x) else "NaN"
        )

    if "p-value AD" in df.columns:
        df["p-value AD"] = pd.to_numeric(df["p-value AD"], errors="coerce")
        # apply scientific notation for very small p-values
        df["p-value AD"] = df["p-value AD"].apply(
            lambda x: f"{x:.0e}" if x < 1e-3 else f"{x:.3f}" if pd.notnull(x) else "NaN"
        )

    # save the DataFrame as a CSV file
    df.to_csv(file_name, index=False)

    print(f"Data has been saved to {file_name}")


def SaveTableAsxlsx(pTable, file_name="output.xlsx"):
    df = pd.DataFrame(pTable)

    # set a default float format for other columns
    pd.options.display.float_format = "{:.2f}".format

    # check if "p-value Dementia" column exists and convert it to numeric if necessary
    if "p-value Dementia" in df.columns:
        df["p-value Dementia"] = pd.to_numeric(df["p-value Dementia"], errors="coerce")
        # apply scientific notation for very small p-values
        df["p-value Dementia"] = df["p-value Dementia"].apply(
            lambda x: f"{x:.0e}" if x < 1e-3 else f"{x:.3f}" if pd.notnull(x) else "NaN"
        )

    if "p-value AD" in df.columns:
        df["p-value AD"] = pd.to_numeric(df["p-value AD"], errors="coerce")
        # apply scientific notation for very small p-values
        df["p-value AD"] = df["p-value AD"].apply(
            lambda x: f"{x:.0e}" if x < 1e-3 else f"{x:.3f}" if pd.notnull(x) else "NaN"
        )

    # Save the DataFrame as a excel file
    df.to_excel(file_name, index=False, engine="openpyxl")

    print(f"Data has been saved to {file_name}")


def PrintTable(pTable):
    df = pd.DataFrame(pTable)

    # set a default float format for other columns
    pd.options.display.float_format = "{:.2f}".format

    # check if 'p_value' column exists and convert it to numeric if necessary
    if "p-value Dementia" in df.columns:
        df["p-value Dementia"] = pd.to_numeric(df["p-value Dementia"], errors="coerce")
        # apply scientific notation for very small p-values
        df["p-value Dementia"] = df["p-value Dementia"].apply(
            lambda x: f"{x:.0e}" if x < 1e-3 else f"{x:.3f}" if pd.notnull(x) else "NaN"
        )

    if "p-value AD" in df.columns:
        df["p-value AD"] = pd.to_numeric(df["p-value AD"], errors="coerce")
        # apply scientific notation for very small p-values
        df["p-value AD"] = df["p-value AD"].apply(
            lambda x: f"{x:.0e}" if x < 1e-3 else f"{x:.3f}" if pd.notnull(x) else "NaN"
        )

    # print the table as a string without the index
    print(df.to_string(index=False))

# Demographic Features

In [ ]:
# age and Sex statistics added to Char_Table
age_without, age_D, age_AD = Add_Age("Age (years), mean,SD", "")
sex_without, sex_D, sex_AD = Add_Category("Sex, n(%)", ["Women", "Men"], "p31", [0, 1])

# age and sex df created
df_age = pd.concat([age_without, age_D, age_AD], ignore_index=True)
df_sex = pd.concat(
    [sex_without[0], sex_without[1], sex_D[0], sex_D[1], sex_AD[0], sex_AD[1]],
    ignore_index=True,
)
df_sex.columns = ["eid", "Sex"]
df_age.columns = ["eid", "Age"]
df_sex.drop_duplicates(subset=["eid"], keep="first", inplace=True)
df_age.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# education statistics added to Char_Table
education_mapping = {
    1: "Higher",  # College or University degree
    2: "Upper secondary",  # A levels/AS levels or equivalent
    3: "Lower secondary",  # O levels/GCSEs or equivalent
    4: "Lower secondary",  # CSEs or equivalent
    5: "Vocational",  # NVQ or HND or HNC or equivalent
    6: "Other",  # Other professional qualifications
    -7: "Do not know",  # Do not know
    -3: "Prefer not to answer",  # Prefer not to answer
}

# values in p6138_i0 are string representations of lists, convert them to actual lists

Inst0_d["p6138_i0"] = (
    Inst0_d["p6138_i0"]
    .replace("nan", np.nan)
    .apply(lambda x: ast.literal_eval(x) if pd.notna(x) else np.nan)
)

Inst1_d["p6138_i1"] = (
    Inst1_d["p6138_i1"]
    .replace("nan", np.nan)
    .apply(lambda x: ast.literal_eval(x) if pd.notna(x) else np.nan)
)

# minimum value represents the highest education level, so we take the minimum of each list
Inst0_d["p6138_i0"] = Inst0_d["p6138_i0"].apply(
    lambda x: min(x) if isinstance(x, list) and x else np.nan
)
Inst1_d["p6138_i1"] = Inst1_d["p6138_i1"].apply(
    lambda x: min(x) if isinstance(x, list) and x else np.nan
)

education_without, education_D, education_AD = Add_Category_multiple(
    "Educational status, n(%)",
    ["Higher", "Upper secondary", "Lower secondary", "Vocational", "Other"],
    [["p6138_i0"], ["p6138_i1"]],
    [[[1]], [[2]], [[3, 4]], [[5]], [[6, -7]]],
    Inst0_d,
    Inst1_d,
)

# df_educaiton created
df_education = pd.concat(
    [
        education_without[0],
        education_without[1],
        education_without[2],
        education_without[3],
        education_without[4],
        education_D[0],
        education_D[1],
        education_D[2],
        education_D[3],
        education_D[4],
        education_AD[0],
        education_AD[1],
        education_AD[2],
        education_AD[3],
        education_AD[4],
    ],
    ignore_index=True,
)
df_education.columns = ["eid", "Educational status"]
df_education.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# diabetes statistics added to Char_Table
# values in p6177_i0 are string representations of lists, convert them to actual lists

Inst0_d["p6177_i0"] = (
    Inst0_d["p6177_i0"]
    .replace("nan", np.nan)
    .apply(lambda x: ast.literal_eval(x) if pd.notna(x) else np.nan)
)

Inst1_d["p6177_i1"] = (
    Inst1_d["p6177_i1"]
    .replace("nan", np.nan)
    .apply(lambda x: ast.literal_eval(x) if pd.notna(x) else np.nan)
)

Inst0_d["p6153_i0"] = (
    Inst0_d["p6153_i0"]
    .replace("nan", np.nan)
    .apply(lambda x: ast.literal_eval(x) if pd.notna(x) else np.nan)
)

Inst1_d["p6153_i1"] = (
    Inst1_d["p6153_i1"]
    .replace("nan", np.nan)
    .apply(lambda x: ast.literal_eval(x) if pd.notna(x) else np.nan)
)


def check_diabetes(row, inst):
    if row["p31"] == 1:
        return check_diabetes_man(row[f"p6177_i{inst}"])
    else:
        return check_diabetes_woman(row[f"p6153_i{inst}"])


Inst0_d["p6177_i0_diabetes"] = Inst0_d.apply(lambda row: check_diabetes(row, 0), axis=1)
Inst1_d["p6177_i1_diabetes"] = Inst1_d.apply(lambda row: check_diabetes(row, 1), axis=1)


# get final diabetes data
Inst0_d["diabetes_i0"] = combine_diabetes(Inst0_d, "p2443_i0", "p6177_i0_diabetes")
Inst1_d["diabetes_i1"] = combine_diabetes(Inst1_d, "p2443_i1", "p6177_i1_diabetes")
diabetes_without, diabetes_D, diabetes_AD = Add_Category_2_Inst(
    "Diabetes, n(%)",
    ["Without diabetes", "With diabetes"],
    [["diabetes_i0"], ["diabetes_i1"]],
    [0, 1],
)

# df_diabetes created
df_diabetes = pd.concat(
    [
        diabetes_without[0],
        diabetes_without[1],
        diabetes_D[0],
        diabetes_D[1],
        diabetes_AD[0],
        diabetes_AD[1],
    ],
    ignore_index=True,
)
df_diabetes.columns = ["eid", "Diabetes"]
df_diabetes.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# Spherical equivalent statistics added to Char_Table
# SE=(C+(S/2))

N3_field_names = [
    "p5086_i0_a0",
    "p5086_i1_a0",
    "p5087_i0_a0",
    "p5087_i1_a0",  # cylinder power
    "p5085_i0_a0",
    "p5085_i1_a0",
    "p5084_i0_a0",
    "p5084_i1_a0",  # spherical_power
]

cylinder_i0_mean = combine_left_right(Inst0_d, "p5086_i0_a0", "p5087_i0_a0")
cylinder_i1_mean = combine_left_right(Inst1_d, "p5086_i1_a0", "p5087_i1_a0")
spherical_i0_mean = combine_left_right(Inst0_d, "p5084_i0_a0", "p5085_i0_a0")
spherical_i1_mean = combine_left_right(Inst1_d, "p5084_i1_a0", "p5085_i1_a0")

spherical_equivalent_i0 = pd.DataFrame()
spherical_equivalent_i0["eid"] = cylinder_i0_mean["eid"]
spherical_equivalent_i1 = pd.DataFrame()
spherical_equivalent_i1["eid"] = cylinder_i1_mean["eid"]

# calculate spherical equivalent
spherical_equivalent_i0["spherical_eq_mean_i0"] = spherical_i0_mean[
    "p5084_i0_a0_mean"
] + (cylinder_i0_mean["p5086_i0_a0_mean"] / 2)
spherical_equivalent_i1["spherical_eq_mean_i1"] = spherical_i1_mean[
    "p5084_i1_a0_mean"
] + (cylinder_i1_mean["p5086_i1_a0_mean"] / 2)
spheric_without, spheric_D, spheric_AD = Add_Category_2_InstIQR(
    spherical_equivalent_i0,
    spherical_equivalent_i1,
    "Spherical equivalent, median, IQR",
    "",
    [["spherical_eq_mean_i0"], ["spherical_eq_mean_i1"]],
)


# df_spheric created
df_spheric = pd.concat([spheric_without, spheric_D, spheric_AD], ignore_index=True)
df_spheric.columns = ["eid", "Spherical equivalent"]
df_spheric.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# systolic blood pressure statistics added to Char_Table
Inst0_d["p4080_i0"] = (Inst0_d["p4080_i0_a0"] + Inst0_d["p4080_i0_a1"]) / 2
Inst1_d["p4080_i1"] = (Inst1_d["p4080_i1_a0"] + Inst1_d["p4080_i1_a1"]) / 2
systolic_without, systolic_D, systolic_AD = Add_Category_2_InstMean(
    Inst0_d,
    Inst1_d,
    "Systolic blood pressure (mmHg), mean,SD",
    "",
    [["p4080_i0"], ["p4080_i1"]],
    [],
)

# df_ststolic created
df_systolic = pd.concat([systolic_without, systolic_D, systolic_AD], ignore_index=True)
df_systolic.columns = ["eid", "Systolic blood pressure"]
df_systolic.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# diastolic blood pressure statistics added to Char_Table
Inst0_d["p4079_i0"] = (Inst0_d["p4079_i0_a0"] + Inst0_d["p4079_i0_a1"]) / 2
Inst1_d["p4079_i1"] = (Inst1_d["p4079_i1_a0"] + Inst1_d["p4079_i1_a1"]) / 2
diastolic_without, diastolic_D, diastolic_AD = Add_Category_2_InstMean(
    Inst0_d,
    Inst1_d,
    "Diastolic blood pressure (mmHg), mean,SD",
    "",
    [["p4079_i0"], ["p4079_i1"]],
    [],
)

# df_diastolic created
df_diastolic = pd.concat(
    [diastolic_without, diastolic_D, diastolic_AD], ignore_index=True
)
df_diastolic.columns = ["eid", "Diastolic blood pressure"]
df_diastolic.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# hypertension statistics added to Char_Table
def check_hypertension(row, inst):
    if row["p31"] == 1:
        return check_hypertension_man(row[f"p6177_i{inst}"])
    else:
        return check_hypertension_woman(row[f"p6153_i{inst}"])


Inst0_d["p6177_i0_hyper"] = Inst0_d.apply(
    lambda row: check_hypertension(row, 0), axis=1
)
Inst1_d["p6177_i1_hyper"] = Inst1_d.apply(
    lambda row: check_hypertension(row, 1), axis=1
)


hypertensive_without, hypertensive_D, hypertensive_AD = Add_Category_2_Inst(
    "Antihypertensive medication use, n(%)",
    ["Without antihypertensive medication", "With antihypertensive medication"],
    [["p6177_i0_hyper"], ["p6177_i1_hyper"]],
    [0, 1],
)

# df_hypertenstion created
df_hypertensive = pd.concat(
    [
        hypertensive_without[0],
        hypertensive_without[1],
        hypertensive_D[0],
        hypertensive_D[1],
        hypertensive_AD[0],
        hypertensive_AD[1],
    ],
    ignore_index=True,
)
df_hypertensive.columns = ["eid", "Antihypertensive usage"]
df_hypertensive.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# alcohol statistics added to Char_Table
alcohol_without, alcohol_D, alcohol_AD = Add_Category_2_Inst(
    "Alcohol Consumption, n(%)",
    ["Daily", "3 or 4 /week", "1 or 2/week", "1-3/month", "Special occations", "Never"],
    [["p1558_i0"], ["p1558_i1"]],
    [1, 2, 3, 4, 5, 6],
)

# alcohol df created
df_alcohol = pd.concat(
    [
        alcohol_without[0],
        alcohol_without[1],
        alcohol_without[2],
        alcohol_without[3],
        alcohol_without[4],
        alcohol_without[5],
        alcohol_D[0],
        alcohol_D[1],
        alcohol_D[2],
        alcohol_D[3],
        alcohol_D[4],
        alcohol_D[5],
        alcohol_AD[0],
        alcohol_AD[1],
        alcohol_AD[2],
        alcohol_AD[3],
        alcohol_AD[4],
        alcohol_AD[5],
    ],
    ignore_index=True,
)
df_alcohol.columns = ["eid", "Alcohol Consumption"]
df_alcohol.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# smoking statistics added to Char_Table
smoking_without, smoking_D, smoking_AD = Add_Category_2_Inst(
    "Smoking Status, n(%)",
    ["Never", "Previous", "Current"],
    [["p20116_i0"], ["p20116_i1"]],
    [0, 1, 2],
)

# df_smoking created
df_smoking = pd.concat(
    [
        smoking_without[0],
        smoking_without[1],
        smoking_without[2],
        smoking_D[0],
        smoking_D[1],
        smoking_D[2],
        smoking_AD[0],
        smoking_AD[1],
        smoking_AD[2],
    ],
    ignore_index=True,
)
df_smoking.columns = ["eid", "Smoking Status"]
df_smoking.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
# body-mass index statistics added to Char_Table
mass_without, mass_D, mass_AD = Add_Category_2_InstMean(
    Inst0_d, Inst1_d, "Body-mass index, mean,SD", "", [["p21001_i0"], ["p21001_i1"]], []
)

# df_mass created
df_mass = pd.concat([mass_without, mass_D, mass_AD], ignore_index=True)
df_mass.columns = ["eid", "BMI"]
df_mass.drop_duplicates(subset=["eid"], keep="first", inplace=True)

# Numeric Eye Features (Extended Features)

In [ ]:
df_take_mean = pd.read_excel(r"./mean.xlsx", header=None)
df_take_mean

In [ ]:
# filter columns ending with '_i0' and '_i1' from df_take_mean
i0_columns = df_take_mean[df_take_mean[0].str.endswith("_i0")][0].tolist()
i1_columns = df_take_mean[df_take_mean[0].str.endswith("_i1")][0].tolist()
i0_columns.append("eid")
i1_columns.append("eid")

# get the rows and columns from Inst0_n and Ins1_n
Inst0_n_filtered = Inst0_n[i0_columns]
Inst1_n_filtered = Inst1_n[i1_columns]

# remove the '_i0' and '_i1' from the column names
Inst0_n_filtered.columns = Inst0_n_filtered.columns.str.replace("_i0", "")
Inst1_n_filtered.columns = Inst1_n_filtered.columns.str.replace("_i1", "")


# Inst1_n_filtered is a subset of Inst0_n_filtered (column wise), I checked it by comparing the columns
# combine the inst0 and inst1

inst_filtered = Inst1_n_filtered[~Inst1_n_filtered["eid"].isin(Inst0_n_filtered["eid"])]
Instn_All = pd.concat([Inst0_n_filtered, inst_filtered], ignore_index=True)
Instn_All

In [ ]:
# get unique column name. This will be used as the column names for the final dataframe
column_names = ["eid"]
for i in df_take_mean[2].str.split("(", expand=True)[0].str.rstrip():
    if i not in column_names:
        column_names.append(i)

mean_numeric_all = calculate_mean_numerical_features(Instn_All, column_names)
mean_numeric_all

In [ ]:
# add mrNFL and mGCIPL thickness statistics to Char_Table
retinal_without, retinal_D, retinal_AD = Add_numeric_mean(
    "Baseline retinal thickness indices",
    ["mRNFL thickness(μm), mean,SD", "mGCIPL thickness(μm), mean,SD"],
    [
        "Average retinal nerve fibre layer thickness",
        "Average ganglion cell-inner plexiform layer thickness",
    ],
    mean_numeric_all,
    AddRow,
)

# df_rnfl and df_gcpl created
df_rnfl = pd.concat(
    [retinal_without[0], retinal_D[0], retinal_AD[0]], ignore_index=True
)
df_gcpl = pd.concat(
    [retinal_without[1], retinal_D[1], retinal_AD[1]], ignore_index=True
)
df_rnfl.columns = ["eid", "mRNFL thickness"]
df_gcpl.columns = ["eid", "mGCIPL thickness"]
df_rnfl.drop_duplicates(subset=["eid"], keep="first", inplace=True)
df_gcpl.drop_duplicates(subset=["eid"], keep="first", inplace=True)

In [ ]:
_, _, _ = Add_numeric_mean(
    "Macular thickness indices",
    [
        "Overall thickness",
        "central subfield",
        "inner inferior subfield",
        "inner nasal subfield",
        "inner superior subfield",
        "inner temporal subfield",
        "outer inferior subfield",
        "outer nasal subfield",
        "outer superior subfield",
        "outer temporal subfield",
    ],
    [
        "Overall macular thickness",
        "Macular thickness at the central subfield",
        "Macular thickness at the inner inferior subfield",
        "Macular thickness at the inner nasal subfield",
        "Macular thickness at the inner superior subfield",
        "Macular thickness at the inner temporal subfield",
        "Macular thickness at the outer inferior subfield",
        "Macular thickness at the outer nasal subfield",
        "Macular thickness at the outer superior subfield",
        "Macular thickness at the outer temporal subfield",
    ],
    mean_numeric_all,
    AddRow_Numeric,
)

_, _, _ = Add_numeric_mean(
    "Macular Volume indices",
    ["Total Macular Volume"],
    ["Total macular volume"],
    mean_numeric_all,
    AddRow_Numeric,
)

_, _, _ = Add_numeric_mean(
    "Retinal Pigment Epithelium indices",
    [
        "Overall thickness",
        "central subfield",
        "inner inferior subfield",
        "inner nasal subfield",
        "inner superior subfield",
        "inner temporal subfield",
        "outer inferior subfield",
        "outer nasal subfield",
        "outer superior subfield",
        "outer temporal subfield",
    ],
    [
        "Overall average retinal pigment epithelium thickness",
        "Retinal pigment epithelium thickness at central subfield",
        "Retinal pigment epithelium thickness at inner inferior subfield",
        "Retinal pigment epithelium thickness at inner nasal subfield",
        "Retinal pigment epithelium thickness at inner superior subfield",
        "Retinal pigment epithelium thickness at inner temporal subfield",
        "Retinal pigment epithelium thickness at outer inferior subfield",
        "Retinal pigment epithelium thickness at outer nasal subfield",
        "Retinal pigment epithelium thickness at outer superior subfield",
        "Retinal pigment epithelium thickness at outer temporal subfield",
    ],
    mean_numeric_all,
    AddRow_Numeric,
)

_, _, _ = Add_numeric_mean(
    "Disc Diameter indices",
    [
        "Mean of vertical disc diameter",
        "Disc diameter after inverse rank normal transformation",
    ],
    [
        "Mean of vertical disc diameter",
        "Disc diameter after inverse rank normal transformation",
    ],
    mean_numeric_all,
    AddRow_Numeric,
)


_, _, _ = Add_numeric_mean(
    "Vertical cup to disc ratio indices",
    [
        "Vertical cup to disc ratio",
        "Vertical cup to disc ratio regressed and transformed",
    ],
    [
        "Vertical cup to disc ratio",
        "Vertical cup to disc ratio regressed and transformed",
    ],
    mean_numeric_all,
    AddRow_Numeric,
)

_, _, _ = Add_numeric_mean(
    "inner nuclear layer indices",
    ["Average inner nuclear layer thickness"],
    ["Average inner nuclear layer thickness"],
    mean_numeric_all,
    AddRow_Numeric,
)


_, _, _ = Add_numeric_mean(
    "Retinal Nerve Fibre Layer indices",
    ["mRNFL thickness"],
    ["Average retinal nerve fibre layer thickness"],
    mean_numeric_all,
    AddRow_Numeric,
)

_, _, _ = Add_numeric_mean(
    "Ganglion Cell-inner Plexiform Layer indices",
    ["mGCIPL thickness"],
    ["Average ganglion cell-inner plexiform layer thickness"],
    mean_numeric_all,
    AddRow_Numeric,
)

_, _, _ = Add_numeric_mean(
    "INL-ELM indices",
    [
        "Average thickness",
        "central subfield thickness",
        "inner subfield thickness",
        "outer subfield thickness",
    ],
    [
        "Average INL-ELM thickness",
        "INL-ELM thickness of the central subfield",
        "INL-ELM thickness of the inner subfield",
        "INL-ELM thickness of the outer subfield",
    ],
    mean_numeric_all,
    AddRow_Numeric,
)

_, _, _ = Add_numeric_mean(
    "ELM-ISOS indices",
    [
        "Average thickness",
        "central subfield thickness",
        "inner subfield thickness",
        "outer subfield thickness",
    ],
    [
        "Average ELM-ISOS thickness",
        "ELM-ISOS thickness of central subfield",
        "ELM-ISOS thickness of inner subfield",
        "ELM-ISOS thickness of outer subfield",
    ],
    mean_numeric_all,
    AddRow_Numeric,
)


_, _, _ = Add_numeric_mean(
    "ISOS-RPE indices",
    [
        "Average thickness",
        "central subfield thickness",
        "inner subfield thickness",
        "outer subfield thickness",
    ],
    [
        "Average ISOS-RPE thickness",
        "ISOS-RPE thickness of central subfield",
        "ISOS-RPE thickness of inner subfield",
        "ISOS-RPE thickness of outer subfield",
    ],
    mean_numeric_all,
    AddRow_Numeric,
)

_, _, _ = Add_numeric_mean(
    "INL-RPE indices",
    [
        "Average thickness",
        "central subfield thickness",
        "inner subfield thickness",
        "outer subfield thickness",
    ],
    [
        "Average INL-RPE thickness",
        "INL-RPE thickness of central subfield",
        "INL-RPE thickness of inner subfield",
        "INL-RPE thickness of outer subfield",
    ],
    mean_numeric_all,
    AddRow_Numeric,
)

In [ ]:
ICD10_vascular = [code for code in fields.ICD10 if code.startswith("F01")]
ad_eids = set(pd_AD["eid"])
# Vascular = in pd_D, NOT in AD, and has a vascular ICD10 code
filtered_vascular = pd_D[
    ~pd_D["eid"].isin(ad_eids)
    & pd_D["p41270"].apply(
        lambda x: any(item in x for item in ICD10_vascular) if x is not None else False
    )
]
# Other = everyone in pd_D who is not AD and not Vascular
filtered_other = pd_D[
    ~pd_D["eid"].isin(ad_eids) & ~pd_D["eid"].isin(filtered_vascular["eid"])
]
# Verify mutually exclusive & exhaustive
assert len(pd_AD) + len(filtered_vascular) + len(filtered_other) == len(pd_D), (
    f"Sum {len(pd_AD) + len(filtered_vascular) + len(filtered_other)} != {len(pd_D)}"
)

Add_Category_disease(
    "Types of dementia, n(%)",
    ["Alzheimer's disease", "Vascular dementia", "Other"],
    [pd_AD, filtered_vascular, filtered_other],
)

In [ ]:
pd_AD

In [ ]:
df_status = pd.DataFrame()
df_status["eid"] = All["eid"]
df_status["Status"] = "healthy"
df_status.loc[df_status["eid"].isin(pd_D["eid"]), "Status"] = "dementia"
df_status.loc[df_status["eid"].isin(pd_AD["eid"]), "Status"] = "AD"

In [ ]:
df_status["Status"].value_counts()

In [ ]:
PrintTable(Char_Table)

In [ ]:
PrintTable(Numeric_Char_Table)

In [ ]:
os.makedirs(r"./results", exist_ok=True)
SaveTableAsCSV(Char_Table, r"./results/char_table.csv")
SaveTableAsxlsx(Char_Table, r"./results/char_table.xlsx")

In [ ]:
SaveTableAsCSV(Numeric_Char_Table, r"./results/numeric_char_table.csv")
SaveTableAsxlsx(Numeric_Char_Table, r"./results/numeric_char_table.xlsx")

# Create images

In [ ]:
def create_table_single(without, dementia, ad, title="None", xlabel=None):
    plt.figure(figsize=(8, 4))
    sns.kdeplot(without, label="Healthy", color="blue")
    sns.kdeplot(dementia, label="Dementia", color="orange")
    sns.kdeplot(ad, label="AD", color="green")

    plt.grid(True)
    plt.title(f"Density Plot of {title} Distributions: Healthy, Dementia, and AD")
    plt.xlabel(xlabel)
    plt.ylabel("Density")
    plt.legend()
    path = r"./results/images/" + title + ".png"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.savefig(path)
    plt.show()

In [ ]:
create_table_single(
    age_without["Age"], age_D["Age"], age_AD["Age"], "Age", "Age (years)"
)

In [ ]:
create_table_single(
    systolic_without["New"],
    systolic_D["New"],
    systolic_AD["New"],
    "Systolic Blood Pressure",
    "Systolic Blood Pressure (mmHg)",
)

In [ ]:
smoking_wo = np.concatenate(
    [
        smoking_without[0]["p20116_i0"],
        smoking_without[1]["p20116_i0"],
        smoking_without[2]["p20116_i0"],
    ]
)
smoking_d = np.concatenate(
    [smoking_D[0]["p20116_i0"], smoking_D[1]["p20116_i0"], smoking_D[2]["p20116_i0"]]
)
smoking_ad = np.concatenate(
    [smoking_AD[0]["p20116_i0"], smoking_AD[1]["p20116_i0"], smoking_AD[2]["p20116_i0"]]
)

In [ ]:
educ_wo = np.concatenate([i["New"] for i in education_without])
educ_d = np.concatenate([i["New"] for i in education_D])
educ_ad = np.concatenate([i["New"] for i in education_AD])

# If values are still numeric codes like 1,2,3,4,5 or 0,1,2,3,4
code_map = {
    0: "Higher",
    1: "Upper secondary",
    2: "Lower secondary",
    3: "Vocational",
    4: "Other",
}

cat_order = ["Higher", "Upper secondary", "Lower secondary", "Vocational", "Other"]
x = np.arange(len(cat_order))
width = 0.25

h = (
    pd.Series(educ_wo)
    .map(code_map)
    .value_counts(normalize=True)
    .reindex(cat_order, fill_value=0)
)
d = (
    pd.Series(educ_d)
    .map(code_map)
    .value_counts(normalize=True)
    .reindex(cat_order, fill_value=0)
)
a = (
    pd.Series(educ_ad)
    .map(code_map)
    .value_counts(normalize=True)
    .reindex(cat_order, fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width, h, width, label="Healthy", color="royalblue")
b2 = ax.bar(x, d, width, label="Dementia", color="orange")
b3 = ax.bar(x + width, a, width, label="AD", color="green")
ax.bar_label(b1, fmt=lambda v: f"%{v * 100:.0f}", fontsize=12, padding=2, color="blue")
ax.bar_label(
    b2, fmt=lambda v: f"%{v * 100:.0f}", fontsize=12, padding=2, color="orange"
)
ax.bar_label(b3, fmt=lambda v: f"%{v * 100:.0f}", fontsize=12, padding=2, color="green")
ax.set_xticks(x)
ax.set_xticklabels(cat_order)
ax.set_ylabel("Proportion")
ax.set_title("Education Distributions: Healthy, Dementia, and AD")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(r"./results/images/education_distribution.png")
plt.show()

In [ ]:
create_table_single(
    spheric_without["New"],
    spheric_D["New"],
    spheric_AD["New"],
    "Spherical Equivalent",
    "Spherical Equivalent (dpt)",
)

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(
    retinal_without[0]["Average retinal nerve fibre layer thickness"],
    label="Healthy",
    color="blue",
)
sns.kdeplot(
    retinal_D[0]["Average retinal nerve fibre layer thickness"],
    label="Dementia",
    color="orange",
)
sns.kdeplot(
    retinal_AD[0]["Average retinal nerve fibre layer thickness"],
    label="AD",
    color="green",
)
plt.grid(True)
plt.title("Density Plot of mRNFL Thickness Distributions: Healthy, Dementia, and AD")
plt.xlabel("Thickness")
plt.ylabel("Density")
plt.legend()
plt.xlim(0, 50)
plt.savefig(r"./results/images/mRNFL.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(
    retinal_without[1]["Average ganglion cell-inner plexiform layer thickness"],
    label="Healthy",
    color="blue",
)
sns.kdeplot(
    retinal_D[1]["Average ganglion cell-inner plexiform layer thickness"],
    label="Dementia",
    color="orange",
)
sns.kdeplot(
    retinal_AD[1]["Average ganglion cell-inner plexiform layer thickness"],
    label="AD",
    color="green",
)
plt.grid(True)
plt.title("Density Plot of mGCIPL Thickness Distributions: Healthy, Dementia, and AD")
plt.xlabel("Thickness")
plt.ylabel("Density")
plt.xlim(40, 100)
plt.legend()
plt.savefig(r"./results/images/mGCIPL.png")
plt.show()

# Create Final DF

In [ ]:
df_merged = (
    df_age.merge(df_sex, on="eid", how="outer")
    .merge(df_education, on="eid", how="outer")
    .merge(df_diabetes, on="eid", how="outer")
    .merge(df_spheric, on="eid", how="outer")
    .merge(df_systolic, on="eid", how="outer")
    .merge(df_diastolic, on="eid", how="outer")
    .merge(df_hypertensive, on="eid", how="outer")
)
df_merged = (
    df_merged.merge(df_alcohol, on="eid", how="outer")
    .merge(df_smoking, on="eid", how="outer")
    .merge(df_mass, on="eid", how="outer")
    .merge(df_rnfl, on="eid", how="outer")
    .merge(df_gcpl, on="eid", how="outer")
)
df_merged = df_merged.merge(mean_numeric_all, on="eid", how="outer").merge(
    df_status, on="eid", how="outer"
)
df_merged.drop_duplicates(subset=["eid"], keep="first", inplace=True)
df_merged.drop(
    [
        "Average retinal nerve fibre layer thickness",
        "Average ganglion cell-inner plexiform layer thickness",
    ],
    axis=1,
    inplace=True,
)

In [ ]:
df_merged["Status"].value_counts()

In [ ]:
for col in df_merged.columns:
    # get the columns with retinal pigment epithelium
    if col.startswith("Retinal pigment epithelium thickness"):
        df_merged.rename(
            columns={
                col: col.replace(
                    "Retinal pigment epithelium thickness", "RPE thickness"
                )
            },
            inplace=True,
        )

df_merged.rename(
    columns={
        "Overall average retinal pigment epithelium thickness": "Average RPE thickness"
    },
    inplace=True,
)

In [ ]:
df_merged

In [ ]:
CAT_FEATURES = [
    "Age",
    "Sex",
    "Educational status",
    "Diabetes",
    "Alcohol Consumption",
    "Smoking Status",
    "Antihypertensive usage",
]
NUM_FEATURES = df_merged.columns.difference(CAT_FEATURES + ["Status", "eid"])


path = r"./results/subgroups/"

os.makedirs(path, exist_ok=True)

df_merged.to_csv(r"./results/df.csv", index=False)

df_sick = df_merged[df_merged["Status"].isin(["AD", "dementia"])][NUM_FEATURES]
df_ad = df_merged[df_merged["Status"] == "AD"][NUM_FEATURES]
df_merged_numerical = df_merged[NUM_FEATURES]

missing_mask_df_sick = df_sick.isnull().any(axis=1)
missing_mask_ad = df_ad.isnull().any(axis=1)
missing_mask_merged = df_merged_numerical.isnull().any(axis=1)

df_sick_missing = df_sick[missing_mask_df_sick]
df_ad_non_missing = df_ad[~missing_mask_ad]
df_missing = df_merged_numerical[missing_mask_merged]

# save to csv
df_sick_missing.to_csv(path + "dementia_missing.csv")
df_ad_non_missing.to_csv(path + "ad_not_missing.csv")
df_merged_numerical.to_csv(path + "full_data_missing.csv")

In [ ]:
df_merged.info()